# Simulazione numerica del collasso gravitazionale stellare — demo Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ValerioPeperoni/stellar-collapse-simulation/blob/main/demo_colab.ipynb)

Questo notebook esegue il progetto **senza alcun setup locale**: clona il repository, installa le dipendenze, e mette a disposizione un menu a tendina + un solo pulsante per lanciare l'intera pipeline (equilibrio iniziale, dinamica del collasso, classificazione del remnant) su una qualunque stella del catalogo, mostrando grafici e animazioni **direttamente qui nel notebook** (non solo salvati su disco).

**Nessuna modifica alla fisica o al codice esistente**: questo notebook chiama esclusivamente le funzioni gia' presenti nel repository (`collasso.pipeline.run_full_simulation`, `collasso.visualization.*`, la stessa funzione di stampa di `scripts/run_simulation.py`) — e' un modo alternativo di eseguire il progetto, non una riscrittura.

Repository completo, documentazione e log di sviluppo: https://github.com/ValerioPeperoni/stellar-collapse-simulation

## Come usare questo notebook

1. Esegui le due celle della sezione **"1. Setup"** qui sotto (una sola volta per sessione — puoi anche usare *Runtime > Esegui tutte le celle*, il setup e' sicuro da rieseguire).
2. Nella sezione **"2. Scegli la stella e avvia"**, seleziona una stella dal menu a tendina e premi il pulsante **▶ Avvia simulazione**.
3. Per provare un'altra stella, cambia la selezione nel menu e premi di nuovo il pulsante — **non serve rieseguire il setup**.

Stelle disponibili: `s15`, `s20`, `s25`, `s30`, `s35`, `s40` (progenitori reali, O'Connor & Ott 2011) e `betelgeuse` (un *proxy* dichiarato — riusa il modello `s20`, non e' una ricostruzione dedicata di Betelgeuse; vedi il README per il dettaglio completo).

**Limiti del modello** (sempre stampati per esteso ad ogni esecuzione, qui solo un riassunto): nessun trasporto di neutrini, nessun vero bounce fisico modellato, gravita' Newtoniana con correzione relativistica approssimata, simmetria sferica (1D). Dettaglio completo nel [README](https://github.com/ValerioPeperoni/stellar-collapse-simulation#vincoli-fisici-e-approssimazioni-dichiarate) e in [VALIDATION.md](https://github.com/ValerioPeperoni/stellar-collapse-simulation/blob/main/VALIDATION.md).

## 1. Setup (esegui una volta sola)

In [ ]:
import os

REPO_URL = "https://github.com/ValerioPeperoni/stellar-collapse-simulation.git"
REPO_DIR = "/content/stellar-collapse-simulation"

if not os.path.isdir(REPO_DIR):
    print("Clono il repository...")
    !git clone -q {REPO_URL} {REPO_DIR}
    print("Fatto.")
else:
    print("Repository gia' presente in questa sessione Colab, salto il clone.")

%cd {REPO_DIR}

print("\nInstallo le dipendenze da requirements.txt (+ ipywidgets per il menu/pulsante)...")
!pip install -q -r requirements.txt
!pip install -q ipywidgets

print("\nSetup completato.")

In [ ]:
import sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Tutte funzioni GIA' esistenti nel repository, nessuna riscritta qui:
from collasso.catalog import load_reference_catalog
from collasso.pipeline import run_full_simulation
from collasso.visualization import animate_collapse, animate_collapse_disk, plot_summary
from scripts.run_simulation import OUTPUT_DIR, _stampa_risultato

print("Import completato. Stelle disponibili nel catalogo:\n")
for p in load_reference_catalog():
    etichetta_proxy = "  [PROXY - vedi README]" if p.nota_proxy else ""
    print(f"  {p.id:<12} massa ZAMS = {p.massa_zams_msun:>5.1f} Msun{etichetta_proxy}")

## 2. Scegli la stella e avvia — un solo click

Esegui la cella qui sotto una volta per far comparire il menu e il pulsante, poi premi **▶ Avvia simulazione** tutte le volte che vuoi (anche cambiando stella ogni volta), senza dover rieseguire nient'altro.

In [ ]:
import ipywidgets as widgets
from IPython.display import Image as IPImage
from IPython.display import clear_output, display

_catalogo = load_reference_catalog()
_opzioni = [
    (p.id + ("  (proxy Betelgeuse)" if p.nota_proxy else ""), p.id)
    for p in _catalogo
]

dropdown_stella = widgets.Dropdown(options=_opzioni, value="s20", description="Stella:")
bottone_avvia = widgets.Button(
    description="Avvia simulazione", button_style="success", icon="play"
)
area_output = widgets.Output()


def _esegui_simulazione(_pulsante):
    with area_output:
        clear_output(wait=True)
        star_id = dropdown_stella.value
        try:
            print(f"Eseguo la pipeline completa (Step 1-6) per '{star_id}'... puo' richiedere qualche decina di secondi.\n")
            result = run_full_simulation(star_id)

            # Stessa funzione usata da `python scripts/run_simulation.py` da riga di comando:
            _stampa_risultato(result)

            print("\nGenero le visualizzazioni...\n")
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            gif_path = animate_collapse(result, OUTPUT_DIR / f"collapse_{star_id}.gif")
            disk_gif_path = animate_collapse_disk(result, OUTPUT_DIR / f"collapse_disk_{star_id}.gif")
            png_path = plot_summary(result, OUTPUT_DIR / f"summary_{star_id}.png")

            print("Animazione 2D (disco colorato per densita' - vedi il testo sull'ultimo fotogramma):")
            display(IPImage(filename=str(disk_gif_path)))

            print("\nAnimazione a linea (raggio di ogni shell vs frazione di massa racchiusa):")
            display(IPImage(filename=str(gif_path)))

            print("\nGrafico riassuntivo (densita' centrale, velocita' shell interna, curva massa-raggio TOV):")
            display(IPImage(filename=str(png_path)))

            print(f"\nCompletato. File salvati anche su disco in: {OUTPUT_DIR}")

        except Exception as errore:
            print(f"\nERRORE durante l'esecuzione: {type(errore).__name__}: {errore}")
            print(
                "Se il setup (sezione 1) non e' stato eseguito prima di premere il "
                "pulsante, esegui prima le due celle di setup e riprova."
            )
            raise


bottone_avvia.on_click(_esegui_simulazione)

display(widgets.HBox([dropdown_stella, bottone_avvia]), area_output)

---

Progetto completo, catalogo dei progenitori, test (105), log di sviluppo e validazione qualitativa contro GR1D: vedi il [README](https://github.com/ValerioPeperoni/stellar-collapse-simulation) del repository.